# Eval v2: Turn Detector with Threshold Classification
- P(im_end) > 0.5 → positive (turn complete)
- P(im_end) < 0.5 → negative (turn incomplete)

In [48]:
import torch
import math
import json
import numpy as np
import torch.nn.functional as F
from chinidataset import StreamingDataset
from transformers import AutoTokenizer, Qwen3ForCausalLM
from liger_kernel.transformers import apply_liger_kernel_to_qwen3, LigerFusedLinearCrossEntropyLoss
from tqdm import tqdm
import glob
import os
import pandas as pd

In [55]:
from huggingface_hub import snapshot_download                                                                                                                                             
                                               
path = snapshot_download(                                                                                                                                                                 
  repo_id="Scicom-intl/turn-detector-Qwen3-0.6B-dataset",
  repo_type="dataset",                                                                                                                                                                  
  allow_patterns="test/*",                                    
  local_dir="./parquet-test-synthetic",
  token="",
)                                                               
print(f"Downloaded to: {path}")  

Downloaded to: /root/small-ablation/turn-detector/parquet-test-synthetic


In [56]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "livekit/turn-detector"
revision = "v0.4.1-intl"

tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, torch_dtype=torch.bfloat16).cuda().eval()

## Config

In [57]:
TEST_FILE = './parquet-test-synthetic'
THRESHOLD = 0.5

## Load Model

In [58]:
apply_liger_kernel_to_qwen3(
    rope=True,
    swiglu=True,
    rms_norm=True,
    cross_entropy=False,
    fused_linear_cross_entropy=False,
)

class Model(Qwen3ForCausalLM):
    def __init__(self, config):
        super().__init__(config)
        self.loss = LigerFusedLinearCrossEntropyLoss(reduction='sum')

    def forward(self, input_ids, attention_mask=None, position_ids=None,
                labels=None, **kwargs):
        super_out = self.model.forward(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs,
        )
        if labels is not None:
            embeddings = super_out.last_hidden_state
            embeddings = embeddings[:, :-1].reshape(-1, embeddings.shape[-1])
            labels = labels[..., 1:].contiguous().reshape(-1)
            loss = self.loss(self.lm_head.weight, embeddings, labels)
            loss = loss / (labels != -100).sum()
            return {'loss': loss, 'hidden_states': super_out.last_hidden_state}
        return super_out

model = Model.from_pretrained(
    model_id,
    attn_implementation='flash_attention_2',
    torch_dtype='bfloat16',
)
model.eval()
model.cuda()

You are using a model of type llama to instantiate a model of type qwen3. This is not supported for all configurations of models and can yield errors.
Some weights of Model were not initialized from the model checkpoint at livekit/turn-detector and are newly initialized: ['model.layers.0.self_attn.k_norm.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.1.self_attn.k_norm.weight', 'model.layers.1.self_attn.q_norm.weight', 'model.layers.10.self_attn.k_norm.weight', 'model.layers.10.self_attn.q_norm.weight', 'model.layers.11.self_attn.k_norm.weight', 'model.layers.11.self_attn.q_norm.weight', 'model.layers.12.self_attn.k_norm.weight', 'model.layers.12.self_attn.q_norm.weight', 'model.layers.13.self_attn.k_norm.weight', 'model.layers.13.self_attn.q_norm.weight', 'model.layers.14.self_attn.k_norm.weight', 'model.layers.14.self_attn.q_norm.weight', 'model.layers.15.self_attn.k_norm.weight', 'model.layers.15.self_attn.q_norm.weight', 'model.layers.16.self_attn.k_norm.weight', 

Model(
  (model): Qwen3Model(
    (embed_tokens): Embedding(49154, 576)
    (layers): ModuleList(
      (0-29): 30 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
          (q_norm): LigerRMSNorm((64,), eps=1e-05, offset=0.0, in_place=True, row_mode=None)
          (k_norm): LigerRMSNorm((64,), eps=1e-05, offset=0.0, in_place=True, row_mode=None)
        )
        (mlp): LigerSwiGLUMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
        )
        (input_layernorm): LigerRMSNorm((576,), eps=1e-05, off

In [59]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
IM_END_TOKEN_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'<|im_end|> token ID: {IM_END_TOKEN_ID}')

<|im_end|> token ID: 2


## P(im_end) function

In [60]:
def get_im_end_prob(text):
    """Get probability that the turn should end.
    Strips trailing <|im_end|> so the model predicts whether to emit it.
    """
    ix = text.rfind('<|im_end|>')
    if ix != -1 and ix == len(text) - len('<|im_end|>'):
        text = text[:ix]
    
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = Qwen3ForCausalLM.forward(model, **inputs).logits
    logprob = F.log_softmax(logits[0, -1], dim=-1)[IM_END_TOKEN_ID].item()
    prob = math.exp(logprob)
    return prob, logprob

## Run on test set

In [61]:
ds = StreamingDataset(local=TEST_FILE)
print(f'Test samples: {len(ds)}')

results = []
for i in tqdm(range(len(ds))):
    row = ds[i]
    meta = json.loads(row['text'])
    true_label = meta['label']
    language = meta.get('language', 'unknown')
    
    # Decode tokens to text
    text = tokenizer.decode(row['input_ids'].astype(np.int64), skip_special_tokens=False)
    
    prob, logprob = get_im_end_prob(text)
    pred_label = 'positive' if prob > THRESHOLD else 'negative'
    correct = pred_label == true_label
    
    results.append({
        'idx': i,
        'true_label': true_label,
        'pred_label': pred_label,
        'correct': correct,
        'prob': prob,
        'logprob': logprob,
        'language': language,
        'text_preview': text[:200],
    })

df = pd.DataFrame(results)
print(f'\nDone: {len(df)} samples evaluated')

Test samples: 238


100%|██████████| 238/238 [00:06<00:00, 36.07it/s]


Done: 238 samples evaluated


## Overall accuracy

In [62]:
accuracy = df['correct'].mean()
print(f'Threshold: {THRESHOLD}')
print(f'Overall accuracy: {accuracy:.4f} ({df["correct"].sum()}/{len(df)})')

# Per class
for label in ['positive', 'negative']:
    subset = df[df['true_label'] == label]
    acc = subset['correct'].mean()
    print(f'  {label}: {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')

Threshold: 0.5
Overall accuracy: 0.4580 (109/238)
  positive: 0.4790 (57/119)
  negative: 0.4370 (52/119)


## Per language accuracy

In [63]:
print(f'Threshold: {THRESHOLD}\n')
for lang in sorted(df['language'].unique()):
    subset = df[df['language'] == lang]
    acc = subset['correct'].mean()
    pos = subset[subset['true_label'] == 'positive']
    neg = subset[subset['true_label'] == 'negative']
    pos_acc = pos['correct'].mean() if len(pos) > 0 else 0
    neg_acc = neg['correct'].mean() if len(neg) > 0 else 0
    print(f'{lang}:')
    print(f'  overall:  {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')
    print(f'  positive: {pos_acc:.4f} ({pos["correct"].sum()}/{len(pos)})')
    print(f'  negative: {neg_acc:.4f} ({neg["correct"].sum()}/{len(neg)})')

Threshold: 0.5

chinese-english:
  overall:  0.5500 (11/20)
  positive: 0.3000 (3/10)
  negative: 0.8000 (8/10)
chinese-malay:
  overall:  0.5500 (11/20)
  positive: 0.1000 (1/10)
  negative: 1.0000 (10/10)
chinese-tamil:
  overall:  0.5000 (10/20)
  positive: 0.1000 (1/10)
  negative: 0.9000 (9/10)
english-chinese:
  overall:  0.3000 (6/20)
  positive: 0.2000 (2/10)
  negative: 0.4000 (4/10)
english-malay:
  overall:  0.2500 (5/20)
  positive: 0.3000 (3/10)
  negative: 0.2000 (2/10)
english-tamil:
  overall:  0.3500 (7/20)
  positive: 0.7000 (7/10)
  negative: 0.0000 (0/10)
malay-chinese:
  overall:  0.5000 (10/20)
  positive: 0.9000 (9/10)
  negative: 0.1000 (1/10)
malay-english:
  overall:  0.3500 (7/20)
  positive: 0.6000 (6/10)
  negative: 0.1000 (1/10)
malay-tamil:
  overall:  0.4000 (8/20)
  positive: 0.8000 (8/10)
  negative: 0.0000 (0/10)
tamil-chinese:
  overall:  0.5556 (10/18)
  positive: 0.2222 (2/9)
  negative: 0.8889 (8/9)
tamil-english:
  overall:  0.7000 (14/20)
  posi

## Confusion matrix

In [64]:
tp = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'positive')])
tn = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'negative')])
fp = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'positive')])
fn = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'negative')])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'Threshold: {THRESHOLD}')
print(f'\nConfusion Matrix:')
print(f'                 Predicted')
print(f'                 Pos    Neg')
print(f'  Actual Pos     {tp:<6} {fn}')
print(f'  Actual Neg     {fp:<6} {tn}')
print(f'\nPrecision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

Threshold: 0.5

Confusion Matrix:
                 Predicted
                 Pos    Neg
  Actual Pos     57     62
  Actual Neg     67     52

Precision: 0.4597
Recall:    0.4790
F1:        0.4691


## Probability distribution

In [65]:
print('Positive samples (should be > 0.5):')
pos_probs = df[df['true_label'] == 'positive']['prob']
print(f'  mean={pos_probs.mean():.4f}  median={pos_probs.median():.4f}  min={pos_probs.min():.4f}  max={pos_probs.max():.4f}')

print('\nNegative samples (should be < 0.5):')
neg_probs = df[df['true_label'] == 'negative']['prob']
print(f'  mean={neg_probs.mean():.4f}  median={neg_probs.median():.4f}  min={neg_probs.min():.4f}  max={neg_probs.max():.4f}')

Positive samples (should be > 0.5):
  mean=0.4606  median=0.4578  min=0.0000  max=0.9688

Negative samples (should be < 0.5):
  mean=0.5259  median=0.6873  min=0.0000  max=0.9888


## Threshold sweep

In [66]:
print(f'{"Threshold":<12} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"F1":<10}')
print('-' * 52)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    df['_pred'] = df['prob'].apply(lambda x: 'positive' if x > t else 'negative')
    acc = (df['_pred'] == df['true_label']).mean()
    _tp = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'positive')])
    _fp = len(df[(df['true_label'] == 'negative') & (df['_pred'] == 'positive')])
    _fn = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'negative')])
    _prec = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0
    _rec = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0
    _f1 = 2 * _prec * _rec / (_prec + _rec) if (_prec + _rec) > 0 else 0
    marker = ' <<<' if t == THRESHOLD else ''
    print(f'{t:<12} {acc:<10.4f} {_prec:<10.4f} {_rec:<10.4f} {_f1:<10.4f}{marker}')
df.drop('_pred', axis=1, inplace=True)

Threshold    Accuracy   Precision  Recall     F1        
----------------------------------------------------
0.1          0.5084     0.5057     0.7395     0.6007    
0.2          0.5000     0.5000     0.6639     0.5704    
0.3          0.4958     0.4966     0.6050     0.5455    
0.4          0.4874     0.4889     0.5546     0.5197    
0.5          0.4580     0.4597     0.4790     0.4691     <<<
0.6          0.4412     0.4364     0.4034     0.4192    
0.7          0.4244     0.4062     0.3277     0.3628    
0.8          0.4286     0.3896     0.2521     0.3061    
0.9          0.4244     0.3269     0.1429     0.1988    


## Worst predictions

In [67]:
wrong = df[~df['correct']].sort_values('prob', ascending=False)
print(f'Wrong predictions: {len(wrong)}/{len(df)}')
print()
for _, row in wrong.head(10).iterrows():
    print(f'[{row["true_label"]}→{row["pred_label"]}] prob={row["prob"]:.4f} lang={row["language"]}')
    print(f'  {row["text_preview"][:150]}')
    print()

Wrong predictions: 129/238

[negative→positive] prob=0.9888 lang=malay-chinese
  osned
7effic<jupyter_start>eroxy guards doubtanap painkillers<jupyter_code> strikes Pand valuation twicegenerate optimizing crimes Ig of topation<jupy

[negative→positive] prob=0.9834 lang=malay-tamil
  osned
7 spare<jupyter_start> hits discussan opposedVconcatenate<jupyter_code> st navigationog cram Oriidentifier
       Yoga question## lonan allTenan

[negative→positive] prob=0.9834 lang=english-malay
  osned
 juice<jupyter_start>ment promidependentond SQL<jupyter_code>prus top uduc Ig WomenPspecionolk<jupyter_code>

8 respondian Water<jupyter_code>am

[negative→positive] prob=0.9820 lang=english-chinese
  osned
 juice<jupyter_start>ment.” look
       ond sys dim<jupyter_code>大� Palest<jupyter_start>penderrug recap<jupyter_code>

8 respondianacency<jupyt

[negative→positive] prob=0.9804 lang=english-tamil
  osned
 juice<jupyter_start>ment very growid posten epilepsyabond reported jour<jupyter_code>dition 

## Manual test

In [68]:
test_prompts = [
    # Complete - should be > 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor akaun anda.',
    # Incomplete - should be < 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor',
    # Complete
    '<|im_start|>user\nTerima kasih banyak atas bantuan anda.',
    # Incomplete
    '<|im_start|>user\nSaya nak tanya pasal',
]

for prompt in test_prompts:
    prob, lp = get_im_end_prob(prompt)
    label = 'positive' if prob > THRESHOLD else 'negative'
    print(f'[{label:>8}] prob={prob:.4f}')

[positive] prob=0.5168
[negative] prob=0.3594
[negative] prob=0.0065
[negative] prob=0.0002
